# 🎥 Gemastik 2026: 5-Fold Stratified GroupKFold Video ViT Classifier
### ⚡ Optimized for NVIDIA L4 GPU (24GB VRAM) in Google Colab
### Pretrained Backbone: `dima806/deepfake_vs_real_image_detection` (Vision Transformer - ViT)

This notebook fine-tunes the Vision Transformer (ViT) model **`dima806/deepfake_vs_real_image_detection`** using **5-Fold Stratified GroupKFold Cross-Validation** on the Indonesian video dataset (`Real/train.zip` and `AI/trainAI.zip`).

### 🚀 NVIDIA L4 GPU Optimizations:
1. **Batch Size 32:** Efficiently utilizes the 24 GB Ada Lovelace VRAM.
2. **Automatic Mixed Precision (AMP):** Fast FP16/BF16 tensor core acceleration (`torch.cuda.amp.autocast`).
3. **DataLoader Optimizations:** `num_workers=4`, `pin_memory=True`, `persistent_workers=True` for high throughput.
4. **VRAM Memory Management:** Explicit `torch.cuda.empty_cache()` cleanup between folds.
5. **Isolated Revision 1 Output Path:** Checkpoints saved to `/content/drive/MyDrive/Gemastik26/models/revision1/`.

## 1. Setup Environment & L4 GPU Hardware Check

In [ ]:
# Install required libraries
!pip install -q transformers timm albumentations opencv-python-headless scikit-learn matplotlib seaborn tqdm pandas pillow

import os
import glob
import shutil
import random
import zipfile
import datetime
import json
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from transformers import AutoImageProcessor, AutoModelForImageClassification

# Hardware Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ PyTorch running on device: {device}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮 Detected GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
    if "L4" in gpu_name:
        print("🚀 NVIDIA L4 GPU detected! High-throughput batching and AMP FP16 active.")

# Optional Colab Mount
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("ℹ️ Local environment detected.")

## 2. Model Backbone & Directory Setup

In [ ]:
# Hyperparameter Directory & Profile Loading
HYPERPARAM_DIR = "notebook/revision1/hyperparameter"
DRIVE_HYPERPARAM_DIR = "/content/drive/MyDrive/Gemastik26/hyperparameter"

# Auto-detect hardware profile for Video Modality
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
profile_filename = "video_colab_L4.json" if "L4" in gpu_name else ("video_colab_T4.json" if "T4" in gpu_name else "video_colab_L4.json")

config_path = os.path.join(HYPERPARAM_DIR, profile_filename)
if not os.path.exists(config_path) and os.path.exists(os.path.join(DRIVE_HYPERPARAM_DIR, profile_filename)):
    config_path = os.path.join(DRIVE_HYPERPARAM_DIR, profile_filename)

if os.path.exists(config_path):
    with open(config_path, "r", encoding="utf-8") as f:
        hp_cfg = json.load(f)
    print(f"✅ LOADED VIDEO HYPERPARAMETERS FROM CONFIG: {config_path}")
else:
    print(f"ℹ️ Config file {config_path} not found. Using default Video L4 GPU profile.")
    hp_cfg = {
        "modality": "video",
        "hf_model_id": "dima806/deepfake_vs_real_image_detection",
        "batch_size": 32, "epochs": 10, "learning_rate": 1e-4, "weight_decay": 1e-2,
        "image_size": [224, 224], "target_fps": 1.0, "max_frames_per_video": 30,
        "num_workers": 4, "pin_memory": True, "persistent_workers": True,
        "early_stopping_patience": 3,
        "model_save_dir": "/content/drive/MyDrive/Gemastik26/models/revision1",
        "kfold_splits_file": "/content/drive/MyDrive/Gemastik26/kfold_splits.json"
    }

# Bind Hyperparameters from Loaded JSON
HF_MODEL_ID = hp_cfg.get("hf_model_id", "dima806/deepfake_vs_real_image_detection")
processor = AutoImageProcessor.from_pretrained(HF_MODEL_ID)
print(f"Loaded AutoImageProcessor for '{HF_MODEL_ID}'")

BATCH_SIZE = hp_cfg.get("batch_size", 32)
EPOCHS = hp_cfg.get("epochs", 10)
LEARNING_RATE = hp_cfg.get("learning_rate", 1e-4)
WEIGHT_DECAY = hp_cfg.get("weight_decay", 1e-2)
IMAGE_SIZE = tuple(hp_cfg.get("image_size", [224, 224]))
TARGET_FPS = hp_cfg.get("target_fps", 1.0)
MAX_FRAMES = hp_cfg.get("max_frames_per_video", 30)
NUM_WORKERS = hp_cfg.get("num_workers", 4 if torch.cuda.is_available() else 2)
PIN_MEMORY = hp_cfg.get("pin_memory", True)
PERSISTENT_WORKERS = hp_cfg.get("persistent_workers", True)
PATIENCE = hp_cfg.get("early_stopping_patience", 3)
MODEL_SAVE_DIR = hp_cfg.get("model_save_dir", "/content/drive/MyDrive/Gemastik26/models/revision1")
KFOLD_SPLIT_FILE = hp_cfg.get("kfold_splits_file", "/content/drive/MyDrive/Gemastik26/kfold_splits.json")

# Dataset Directory Paths
DRIVE_DATASET_DIR = "/content/drive/MyDrive/Gemastik26/Dataset Indonesia"
LOCAL_RAW_DIR = "/content/dataset_raw"
LOCAL_FRAMES_DIR = "/content/dataset_frames"
CLASSES = ["Real", "AI"]
N_SPLITS = 5

ZIP_PATHS = {
    "Real": os.path.join(DRIVE_DATASET_DIR, "Real", "train.zip"),
    "AI": os.path.join(DRIVE_DATASET_DIR, "AI", "trainAI.zip")
}

os.makedirs(LOCAL_RAW_DIR, exist_ok=True)
os.makedirs(LOCAL_FRAMES_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

print(f"⚙️ ACTIVE VIDEO HYPERPARAMETERS:")
print(f"   Backbone: {HF_MODEL_ID} | Batch Size: {BATCH_SIZE} | Learning Rate: {LEARNING_RATE} | Epochs: {EPOCHS}")
print(f"   Workers: {NUM_WORKERS} | Image Size: {IMAGE_SIZE} | Model Save Path: {MODEL_SAVE_DIR}")


## 3. Unzip Video Archives & Frame Extraction (1 FPS)

In [ ]:
def unzip_file(zip_path, extract_to):
    if not os.path.exists(zip_path):
        print(f"⚠️ Warning: Zip file not found at {zip_path}")
        return False
    print(f"📦 Unzipping {zip_path} -> {extract_to} ...")
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"✅ Unzipped successfully to {extract_to}")
    return True

for cls in CLASSES:
    extract_target = os.path.join(LOCAL_RAW_DIR, cls)
    zip_p = ZIP_PATHS.get(cls, "")
    if os.path.exists(extract_target) and len(os.listdir(extract_target)) > 0:
        print(f"⏭️ Target directory '{extract_target}' already exists. Skipping unzip.")
    else:
        unzip_file(zip_p, extract_target)

def extract_frames(video_path, output_dir, target_fps=1.0, max_frames=30):
    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0

    video_fps = cap.get(cv2.CAP_PROP_FPS)
    if video_fps <= 0 or np.isnan(video_fps):
        video_fps = 30.0

    frame_interval = max(1, int(round(video_fps / target_fps)))
    count = 0
    saved_count = 0
    video_name = Path(video_path).stem

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if count % frame_interval == 0:
            frame_name = f"{video_name}_frame_{saved_count:04d}.jpg"
            frame_path = os.path.join(output_dir, frame_name)
            cv2.imwrite(frame_path, frame)
            saved_count += 1
            if saved_count >= max_frames:
                break
        count += 1

    cap.release()
    return saved_count

for cls in CLASSES:
    cls_raw_dir = os.path.join(LOCAL_RAW_DIR, cls)
    cls_frames_dir = os.path.join(LOCAL_FRAMES_DIR, cls)
    video_extensions = ("*.mp4", "*.avi", "*.mov", "*.mkv", "*.MP4", "*.AVI", "*.MOV", "*.MKV")
    video_files = []
    for ext in video_extensions:
        video_files.extend(glob.glob(os.path.join(cls_raw_dir, "**", ext), recursive=True))
    
    print(f"🎥 Extracting 1 FPS frames for {len(video_files)} {cls} videos ...")
    for vid in tqdm(video_files, desc=f"Frames [{cls}]"):
        v_stem = Path(vid).stem
        out_v_dir = os.path.join(cls_frames_dir, v_stem)
        if os.path.exists(out_v_dir) and len(os.listdir(out_v_dir)) > 0:
            continue
        extract_frames(vid, out_v_dir, target_fps=1.0, max_frames=30)

print("✅ Frame extraction complete!")

## 4. Load 5-Fold Stratified GroupKFold Split (`kfold_splits.json`)

In [ ]:
all_video_samples = []
for label_idx, cls in enumerate(CLASSES):
    cls_dir = os.path.join(LOCAL_FRAMES_DIR, cls)
    if not os.path.exists(cls_dir):
        continue
    v_folders = [os.path.join(cls_dir, d) for d in os.listdir(cls_dir) if os.path.isdir(os.path.join(cls_dir, d))]
    for vf in v_folders:
        frames = glob.glob(os.path.join(vf, "*.jpg"))
        if frames:
            all_video_samples.append((vf, frames, label_idx, Path(vf).name))

kfold_splits = {}
if os.path.exists(KFOLD_SPLIT_FILE):
    with open(KFOLD_SPLIT_FILE, "r") as f:
        kfold_splits = json.load(f)
    print(f"✅ Loaded 5-Fold split dictionary from {KFOLD_SPLIT_FILE}")
else:
    print(f"⚠️ {KFOLD_SPLIT_FILE} not found! Fallback: Generating 5-Fold split on-the-fly ...")
    from sklearn.model_selection import StratifiedGroupKFold
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    stems = [x[3] for x in all_video_samples]
    labels = [x[2] for x in all_video_samples]
    for fold, (tr_i, val_i) in enumerate(sgkf.split(stems, labels, groups=stems), start=1):
        kfold_splits[f"fold_{fold}"] = {
            "train_stems": list(set([stems[i] for i in tr_i])),
            "val_stems": list(set([stems[i] for i in val_i]))
        }
    with open(KFOLD_SPLIT_FILE, "w") as f:
        json.dump(kfold_splits, f, indent=2)
    print(f"💾 Saved generated 5-Fold split to {KFOLD_SPLIT_FILE}")

print(f"📊 Active Folds Available: {list(kfold_splits.keys())}")

## 5. PyTorch Dataset & L4-Optimized Transforms

In [ ]:
class VideoFrameDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

mean = processor.image_mean if hasattr(processor, 'image_mean') else [0.5, 0.5, 0.5]
std = processor.image_std if hasattr(processor, 'image_std') else [0.5, 0.5, 0.5]

raw_train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

aug_train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

## 6. L4-Optimized 5-Fold Training Pipeline (`train_vit_kfold_pipeline`)

In [ ]:
def train_vit_kfold_pipeline(model_tag="AUGMENTED_DATA", train_transform_func=aug_train_transform, epochs=EPOCHS, patience=3, force_retrain=False):
    fold_results = []
    all_fold_histories = {}
    
    print(f"========================================================")
    print(f"🚀 STARTING 5-FOLD CROSS-VALIDATION FINE-TUNING [{model_tag}]")
    print(f"   Backbone: {HF_MODEL_ID} | Batch Size: {BATCH_SIZE}")
    print(f"========================================================")
    
    for fold_name, split_info in kfold_splits.items():
        print(f"\n--- 🔄 Processing {fold_name.upper()} ---")
        train_stems = set(split_info["train_stems"])
        val_stems = set(split_info["val_stems"])
        
        train_vids = [x for x in all_video_samples if x[3] in train_stems]
        val_vids = [x for x in all_video_samples if x[3] in val_stems]
        
        train_samples = [(f, label) for _, frames, label, _ in train_vids for f in frames]
        val_samples = [(f, label) for _, frames, label, _ in val_vids for f in frames]
        
        save_folder_name = f"dima806_deepfake_aug_{fold_name}"
        target_save_path = os.path.join(MODEL_SAVE_DIR, save_folder_name)
        
        if os.path.exists(target_save_path) and not force_retrain:
            print(f"📂 Existing model found at '{target_save_path}'. Loading & evaluating...")
            eval_model = AutoModelForImageClassification.from_pretrained(target_save_path).to(device)
            eval_model.eval()
            val_ds = VideoFrameDataset(val_samples, transform=val_transform)
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
            
            preds_list, targets_list = [], []
            with torch.no_grad():
                for imgs, lbls in tqdm(val_loader, desc=f"[{fold_name}] Eval"):
                    imgs = imgs.to(device)
                    outputs = eval_model(imgs)
                    logits = outputs.logits if hasattr(outputs, 'logits') else outputs
                    _, preds = torch.max(logits, 1)
                    preds_list.extend(preds.cpu().numpy())
                    targets_list.extend(lbls.numpy())
            
            acc = accuracy_score(targets_list, preds_list)
            prec, rec, f1, _ = precision_recall_fscore_support(targets_list, preds_list, average="macro", zero_division=0)
            fold_results.append({
                "Fold": fold_name,
                "Accuracy": acc,
                "Precision": prec,
                "Recall": rec,
                "Macro_F1": f1,
                "Save_Path": target_save_path
            })
            continue
        
        # Load Fresh ViT Model for Fold
        model = AutoModelForImageClassification.from_pretrained(
            HF_MODEL_ID,
            num_labels=len(CLASSES),
            ignore_mismatched_sizes=True,
            id2label={0: "Real", 1: "AI"},
            label2id={"Real": 0, "AI": 1}
        ).to(device)
        
        train_ds = VideoFrameDataset(train_samples, transform=train_transform_func)
        val_ds = VideoFrameDataset(val_samples, transform=val_transform)
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        scaler = torch.cuda.amp.GradScaler()
        
        best_val_loss = float('inf')
        epochs_no_improve = 0
        history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
        
        for epoch in range(1, epochs + 1):
            model.train()
            running_loss, correct, total = 0.0, 0, 0
            pbar = tqdm(train_loader, desc=f"[{fold_name}] Epoch {epoch}/{epochs} Train")
            for imgs, lbls in pbar:
                imgs, lbls = imgs.to(device), lbls.to(device)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast():
                    outputs = model(imgs)
                    logits = outputs.logits if hasattr(outputs, 'logits') else outputs
                    loss = criterion(logits, lbls)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                running_loss += loss.item() * imgs.size(0)
                _, preds = torch.max(logits, 1)
                correct += torch.sum(preds == lbls.data).item()
                total += lbls.size(0)
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
            ep_tr_loss = running_loss / total
            ep_tr_acc = correct / total
            history["train_loss"].append(ep_tr_loss)
            history["train_acc"].append(ep_tr_acc)
            
            # Validation Phase
            model.eval()
            val_loss_sum, val_corr, val_tot = 0.0, 0, 0
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    with torch.cuda.amp.autocast():
                        outputs = model(imgs)
                        logits = outputs.logits if hasattr(outputs, 'logits') else outputs
                        loss = criterion(logits, lbls)
                    val_loss_sum += loss.item() * imgs.size(0)
                    _, preds = torch.max(logits, 1)
                    val_corr += torch.sum(preds == lbls.data).item()
                    val_tot += lbls.size(0)
            
            ep_val_loss = val_loss_sum / val_tot
            ep_val_acc = val_corr / val_tot
            history["val_loss"].append(ep_val_loss)
            history["val_acc"].append(ep_val_acc)
            scheduler.step()
            
            print(f"[{fold_name}] Ep {epoch:02d}/{epochs:02d} | Train Loss: {ep_tr_loss:.4f} Acc: {ep_tr_acc*100:.2f}% | Val Loss: {ep_val_loss:.4f} Acc: {ep_val_acc*100:.2f}%")
            
            if ep_val_loss < best_val_loss:
                best_val_loss = ep_val_loss
                epochs_no_improve = 0
                model.save_pretrained(target_save_path)
                processor.save_pretrained(target_save_path)
                print(f"  🏆 Saved best model for {fold_name} to: {target_save_path}")
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f"  🛑 Early stopping triggered for {fold_name} at epoch {epoch}.")
                    break
        
        all_fold_histories[fold_name] = history
        
        # Evaluate best fold checkpoint
        eval_model = AutoModelForImageClassification.from_pretrained(target_save_path).to(device)
        eval_model.eval()
        preds_list, targets_list = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs = imgs.to(device)
                outputs = eval_model(imgs)
                logits = outputs.logits if hasattr(outputs, 'logits') else outputs
                _, preds = torch.max(logits, 1)
                preds_list.extend(preds.cpu().numpy())
                targets_list.extend(lbls.numpy())
        
        acc = accuracy_score(targets_list, preds_list)
        prec, rec, f1, _ = precision_recall_fscore_support(targets_list, preds_list, average="macro", zero_division=0)
        fold_results.append({
            "Fold": fold_name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "Macro_F1": f1,
            "Save_Path": target_save_path
        })
        
        # Clean VRAM cache between folds
        del model, eval_model
        torch.cuda.empty_cache()
    
    return fold_results, all_fold_histories

## 7. Run 5-Fold ViT Training & Fine-Tuning Execution

In [ ]:
fold_results, fold_histories = train_vit_kfold_pipeline(
    model_tag="AUGMENTED_DATA",
    train_transform_func=aug_train_transform,
    epochs=EPOCHS,
    patience=3,
    force_retrain=False
)

## 8. 5-Fold Cross-Validation Metrics & Statistical Summary

In [ ]:
df_kfold = pd.DataFrame(fold_results)

# Compute Mean & Std across 5 Folds
mean_acc = df_kfold["Accuracy"].mean() * 100
std_acc = df_kfold["Accuracy"].std() * 100
mean_f1 = df_kfold["Macro_F1"].mean() * 100
std_f1 = df_kfold["Macro_F1"].std() * 100
mean_prec = df_kfold["Precision"].mean() * 100
std_prec = df_kfold["Precision"].std() * 100
mean_rec = df_kfold["Recall"].mean() * 100
std_rec = df_kfold["Recall"].std() * 100

print("===============================================================")
print("📊 5-FOLD STRATIFIED GROUPKFOLD CROSS-VALIDATION SUMMARY (ViT)")
print("===============================================================")
display(df_kfold)

print("\n--- 📈 OVERALL 5-FOLD CROSS-VALIDATION RESULTS (MEAN ± STD) ---")
print(f"Accuracy:         {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"Precision (Macro): {mean_prec:.2f}% ± {std_prec:.2f}%")
print(f"Recall (Macro):    {mean_rec:.2f}% ± {std_rec:.2f}%")
print(f"Macro F1-Score:   {mean_f1:.2f}% ± {std_f1:.2f}%")

## 9. Single Video 5-Fold Ensemble Inference Helper

In [ ]:
def predict_video_kfold_ensemble(video_path, target_fps=1.0, max_frames=30):
    """
    Ensemble inference across all 5 fine-tuned fold models for maximum generalization.
    """
    frames = extract_frames(video_path, "/content/temp_infer_frames", target_fps=target_fps, max_frames=max_frames)
    frame_paths = glob.glob("/content/temp_infer_frames/*.jpg")
    if not frame_paths:
        return "Failed to extract frames", 0.0
    
    tensors = torch.stack([val_transform(Image.open(fp).convert('RGB')) for fp in frame_paths]).to(device)
    fold_probs = []
    
    for fold_name in kfold_splits.keys():
        model_path = os.path.join(MODEL_SAVE_DIR, f"dima806_deepfake_aug_{fold_name}")
        if os.path.exists(model_path):
            m = AutoModelForImageClassification.from_pretrained(model_path).to(device)
            m.eval()
            with torch.no_grad():
                outputs = m(tensors)
                probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
                fold_probs.append(np.mean(probs, axis=0))
            del m
            torch.cuda.empty_cache()
    
    if not fold_probs:
        return "No fold models found", 0.0
    
    ensemble_prob = np.mean(fold_probs, axis=0)
    pred_idx = int(np.argmax(ensemble_prob))
    pred_label = CLASSES[pred_idx]
    conf = float(ensemble_prob[pred_idx])
    return pred_label, conf

# Example Usage:
# label, conf = predict_video_kfold_ensemble("/content/dataset_raw/AI/sample_video.mp4")